# 06 — Explainability, Ablations & Statistical Validation

**Phases 10, 11, 12 and 13.** This notebook turns the numbers from 03–05 into
claims a reviewer will accept:

1. **Explainability** — Grad-CAM on frames, MIL attention over a study,
   token attribution on the text model, and retrieval-based explanation
   ("this prediction was made because these three past cases looked like it").
2. **Ablations** — what each component actually contributes.
3. **Statistics** — bootstrap confidence intervals, McNemar tests between
   models, and a permutation test against chance.

### Why the statistics section is not optional here

With 259 studies, a 4-point F1 difference between two models is well inside
noise. A paper that reports "our method achieves 0.72 vs 0.68 for the
baseline" without an interval invites exactly one review comment. Every
comparison below comes with a CI and a p-value, and where the difference is
not significant this notebook says so rather than rounding it away.

In [1]:
import os
import sys
import json
import re
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

SEED = 42
rng = np.random.default_rng(SEED)

ART = Path("/kaggle/working/artifacts")
FIG = ART / "figures"
EMB = ART / "embeddings"
for d in (ART, FIG, EMB):
    d.mkdir(parents=True, exist_ok=True)


def find_file(fname):
    for base in (ART, EMB):
        if (base / fname).exists():
            return base / fname
    for p in Path("/kaggle/input").rglob(fname):
        return p
    return None


df = pd.read_csv(find_file("master_split.csv"))
tiers = json.loads(find_file("label_tiers.json").read_text())
Ydf = pd.read_csv(find_file("labels_multihot.csv"))
TARGETS = [t for t in tiers["trainable"] + tiers["marginal"] if t in Ydf.columns]
Y = Ydf[TARGETS].values.astype(np.float32)
folds = df["fold"].values
study_ids = df["study_id"].values
print(f"{len(df)} studies | {len(TARGETS)} targets")

259 studies | 13 targets


## 1. Collect every out-of-fold prediction

Notebooks 03 and 04 saved `oof_<model>.npy`. Significance testing must use
these stored predictions — never re-run a model to compare it, because the
retraining noise will swamp the effect you are testing.

In [2]:
oof_files = sorted(set(
    list(ART.glob("oof_*.npy")) +
    [p for p in Path("/kaggle/input").rglob("oof_*.npy")]))
MODELS = {}
for p in oof_files:
    name = p.stem.replace("oof_", "")
    arr = np.load(p)
    if arr.shape == Y.shape:
        MODELS[name] = arr
    else:
        print(f"  skipping {name}: shape {arr.shape} != {Y.shape}")
print("Loaded models:", list(MODELS))
assert MODELS, "No oof_*.npy found — run notebooks 03 and 04 first."

Loaded models: ['clinicalbert', 'fusion_late', 'img_densenet121', 'img_effnetv2s', 'img_resnet50', 'majority', 'prior', 'tfidf_logreg', 'tfidf_svm', 'multitask', 'probe_concat', 'probe_image', 'probe_text_findings', 'zeroshot']


In [3]:
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score


def tune_thresholds(y_true, y_prob, grid=np.arange(0.05, 0.95, 0.025)):
    th = np.full(y_true.shape[1], 0.5)
    for j in range(y_true.shape[1]):
        if y_true[:, j].sum() == 0:
            continue
        best, bt = -1.0, 0.5
        for t in grid:
            f = f1_score(y_true[:, j], (y_prob[:, j] >= t).astype(int), zero_division=0)
            if f > best:
                best, bt = f, t
        th[j] = bt
    return th


def fold_honest_preds(prob):
    """Binary predictions using thresholds fitted on training folds only."""
    pred = np.zeros_like(prob)
    for f in sorted(set(folds)):
        tr, te = folds != f, folds == f
        th = tune_thresholds(Y[tr], prob[tr])
        pred[te] = (prob[te] >= th).astype(float)
    return pred


PRED = {}
for name, prob in MODELS.items():
    saved = ART / f"thresholds_{name}.npy"
    PRED[name] = fold_honest_preds(prob)
print("Binarised", len(PRED), "models with training-fold thresholds.")

Binarised 14 models with training-fold thresholds.


## 2. Bootstrap confidence intervals

Studies are resampled with replacement 2,000 times; the 2.5th and 97.5th
percentiles give the 95% interval. Resample **studies**, not predictions:
resampling rows of the label matrix independently would break the
correlation between a study's labels and understate the interval.

In [4]:
N_BOOT = 2000


def metric_micro_f1(y, p):
    return f1_score(y, p, average="micro", zero_division=0)


def metric_macro_f1(y, p):
    return f1_score(y, p, average="macro", zero_division=0)


def metric_macro_auc(y, prob):
    a = [roc_auc_score(y[:, j], prob[:, j]) for j in range(y.shape[1])
         if 0 < y[:, j].sum() < len(y)]
    return float(np.mean(a)) if a else np.nan


def bootstrap_ci(fn, y, p, n=N_BOOT, seed=SEED):
    r = np.random.default_rng(seed)
    n_obs = len(y)
    vals = []
    for _ in range(n):
        idx = r.integers(0, n_obs, n_obs)
        if y[idx].sum() == 0:
            continue
        try:
            vals.append(fn(y[idx], p[idx]))
        except ValueError:
            continue
    vals = np.array([v for v in vals if np.isfinite(v)])
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


rows = []
for name in MODELS:
    m1, lo1, hi1 = bootstrap_ci(metric_micro_f1, Y, PRED[name])
    m2, lo2, hi2 = bootstrap_ci(metric_macro_f1, Y, PRED[name])
    m3, lo3, hi3 = bootstrap_ci(metric_macro_auc, Y, MODELS[name])
    rows.append({
        "model": name,
        "micro_F1": metric_micro_f1(Y, PRED[name]),
        "micro_F1_95CI": f"[{lo1:.3f}, {hi1:.3f}]",
        "macro_F1": metric_macro_f1(Y, PRED[name]),
        "macro_F1_95CI": f"[{lo2:.3f}, {hi2:.3f}]",
        "macro_AUC": metric_macro_auc(Y, MODELS[name]),
        "macro_AUC_95CI": f"[{lo3:.3f}, {hi3:.3f}]",
    })
ci_df = pd.DataFrame(rows).sort_values("macro_F1", ascending=False)
for c in ("micro_F1", "macro_F1", "macro_AUC"):
    ci_df[c] = ci_df[c].round(4)
ci_df.to_csv(ART / "results_with_ci.csv", index=False)
print(ci_df.to_string(index=False))

              model  micro_F1  micro_F1_95CI  macro_F1  macro_F1_95CI  macro_AUC macro_AUC_95CI
          multitask    0.8346 [0.788, 0.878]    0.7970 [0.703, 0.853]     0.9452 [0.909, 0.975]
       tfidf_logreg    0.6712 [0.626, 0.718]    0.6361 [0.563, 0.688]     0.9230 [0.898, 0.947]
        fusion_late    0.6328 [0.588, 0.677]    0.6043 [0.526, 0.660]     0.9059 [0.875, 0.934]
          tfidf_svm    0.7111 [0.663, 0.757]    0.5840 [0.522, 0.633]     0.8896 [0.856, 0.925]
       clinicalbert    0.4736 [0.432, 0.514]    0.4503 [0.394, 0.491]     0.8549 [0.816, 0.892]
probe_text_findings    0.3404 [0.310, 0.371]    0.3353 [0.288, 0.372]     0.7979 [0.761, 0.830]
      img_effnetv2s    0.3909 [0.347, 0.434]    0.2929 [0.240, 0.335]     0.7003 [0.653, 0.747]
       probe_concat    0.3233 [0.289, 0.359]    0.2915 [0.252, 0.324]     0.7854 [0.749, 0.820]
    img_densenet121    0.3593 [0.320, 0.399]    0.2801 [0.233, 0.320]     0.6944 [0.644, 0.745]
        probe_image    0.2391 [0.214, 0.

In [5]:
fig, ax = plt.subplots(figsize=(8, 0.45 * len(ci_df) + 2))
order = ci_df.sort_values("macro_F1")
lo = [float(s.strip("[]").split(",")[0]) for s in order["macro_F1_95CI"]]
hi = [float(s.strip("[]").split(",")[1]) for s in order["macro_F1_95CI"]]
y_pos = np.arange(len(order))
ax.errorbar(order["macro_F1"], y_pos,
            xerr=[order["macro_F1"] - lo, hi - order["macro_F1"]],
            fmt="o", capsize=4, color="#4c72b0")
ax.set_yticks(y_pos)
ax.set_yticklabels(order["model"])
ax.set_xlabel("macro-F1 (95% bootstrap CI)")
ax.set_title("Model comparison with uncertainty")
ax.grid(axis="x", alpha=0.3)
plt.savefig(FIG / "results_with_ci.png")
plt.show()

## 3. McNemar tests

McNemar's test compares two classifiers on the *same* cases. It only looks at
disagreements: cases where A is right and B is wrong, and vice versa. For
multi-label output we apply it to the flattened per-(study, label) decision,
with the exact binomial version because discordant counts here are small.

In [6]:
from math import comb


def mcnemar_exact(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    n = b + c
    if n == 0:
        return b, c, 1.0
    k = min(b, c)
    p = sum(comb(n, i) for i in range(k + 1)) / (2 ** n) * 2
    return b, c, float(min(p, 1.0))


names = list(MODELS)

# Unit of analysis = the study. Flattening all 259 x 13 label decisions into
# one vector treats them as independent observations, which they are not:
# labels within a study are correlated, so the flattened test reports p-values
# of ~1e-300 for differences that are not remotely that certain. The primary
# test below uses "did the model get this study's entire label set right",
# giving 259 genuinely independent units. The flattened version is kept as a
# secondary, label-level view — report both, lead with the study-level one.
correct = {n: (PRED[n] == Y).all(axis=1) for n in names}          # study level
correct_flat = {n: (PRED[n] == Y).astype(bool).ravel() for n in names}

pairs = []
for i, a in enumerate(names):
    for b_ in names[i + 1:]:
        nb, nc, p = mcnemar_exact(correct[a], correct[b_])
        _, _, p_flat = mcnemar_exact(correct_flat[a], correct_flat[b_])
        pairs.append({"model_A": a, "model_B": b_,
                      "A_right_B_wrong": nb, "B_right_A_wrong": nc,
                      "p_value": p, "p_value_label_level": p_flat,
                      "better": a if nb > nc else (b_ if nc > nb else "tie"),
                      "significant_0.05": p < 0.05})
mc = pd.DataFrame(pairs).sort_values("p_value")
mc.to_csv(ART / "mcnemar_tests.csv", index=False)
mc_show = mc.copy()
for c in ("p_value", "p_value_label_level"):
    mc_show[c] = mc_show[c].map(lambda v: f"{v:.2e}" if v < 1e-3 else f"{v:.4f}")
print(mc_show.to_string(index=False))

            model_A             model_B  A_right_B_wrong  B_right_A_wrong  p_value p_value_label_level              better  significant_0.05
          multitask         probe_image              206                0 1.94e-62           4.99e-240           multitask              True
              prior           multitask                0              206 1.94e-62            0.00e+00           multitask              True
       img_resnet50           multitask                1              199 2.50e-58           1.36e-227           multitask              True
          multitask            zeroshot              194                3 1.27e-53           2.26e-290           multitask              True
          multitask        probe_concat              168                0 5.35e-51           5.70e-125           multitask              True
          multitask probe_text_findings              175                2 1.64e-49           1.08e-134           multitask              True
    img_dense

### Multiple-comparison correction

Running every pair means many tests. Holm–Bonferroni controls the family-wise
error rate without being as brutal as plain Bonferroni; report corrected
p-values for any comparison you claim in the abstract.

In [7]:
raw_p = pd.read_csv(ART / "mcnemar_tests.csv")["p_value"].astype(float).values
order_i = np.argsort(raw_p)
m = len(raw_p)
holm = np.empty(m)
running = 0.0
for rank, i in enumerate(order_i):
    val = (m - rank) * raw_p[i]
    running = max(running, val)
    holm[i] = min(running, 1.0)
mc_corr = pd.read_csv(ART / "mcnemar_tests.csv")
mc_corr["p_holm"] = holm.round(4)
mc_corr["significant_holm"] = mc_corr["p_holm"] < 0.05
mc_corr.to_csv(ART / "mcnemar_tests.csv", index=False)
print(mc_corr[["model_A", "model_B", "p_value", "p_holm", "significant_holm"]].to_string(index=False))

            model_A             model_B      p_value  p_holm  significant_holm
          multitask         probe_image 1.944692e-62  0.0000              True
              prior           multitask 1.944692e-62  0.0000              True
       img_resnet50           multitask 2.501652e-58  0.0000              True
          multitask            zeroshot 1.268892e-53  0.0000              True
          multitask        probe_concat 5.345529e-51  0.0000              True
          multitask probe_text_findings 1.644794e-49  0.0000              True
    img_densenet121           multitask 2.963989e-46  0.0000              True
              prior           tfidf_svm 4.484155e-44  0.0000              True
          tfidf_svm         probe_image 4.484155e-44  0.0000              True
       tfidf_logreg         probe_image 5.877472e-39  0.0000              True
              prior        tfidf_logreg 5.877472e-39  0.0000              True
       clinicalbert           multitask 1.239779e-38

## 4. Permutation test against chance

Shuffle the labels 1,000 times and recompute macro-F1. The fraction of
shuffles that beat the real score is an empirical p-value. For rare classes
this is far more trustworthy than an asymptotic test.

In [8]:
N_PERM = 1000
perm_rows = []
for name in MODELS:
    obs = metric_macro_f1(Y, PRED[name])
    null = []
    r = np.random.default_rng(SEED)
    for _ in range(N_PERM):
        perm = r.permutation(len(Y))
        null.append(metric_macro_f1(Y[perm], PRED[name]))
    null = np.array(null)
    p = (np.sum(null >= obs) + 1) / (N_PERM + 1)
    perm_rows.append({"model": name, "macro_F1": round(obs, 4),
                      "null_mean": round(float(null.mean()), 4),
                      "null_95pct": round(float(np.percentile(null, 95)), 4),
                      "p_permutation": p,
                      "above_chance": p < 0.05})
perm_df = pd.DataFrame(perm_rows).sort_values("macro_F1", ascending=False)
perm_df.to_csv(ART / "permutation_tests.csv", index=False)
print(perm_df.to_string(index=False))

              model  macro_F1  null_mean  null_95pct  p_permutation  above_chance
          multitask    0.7970     0.0761      0.1053       0.000999          True
       tfidf_logreg    0.6361     0.0795      0.1087       0.000999          True
        fusion_late    0.6043     0.0812      0.1107       0.000999          True
          tfidf_svm    0.5840     0.0782      0.1071       0.000999          True
       clinicalbert    0.4503     0.0858      0.1142       0.000999          True
probe_text_findings    0.3353     0.0966      0.1200       0.000999          True
      img_effnetv2s    0.2929     0.0913      0.1156       0.000999          True
       probe_concat    0.2915     0.0940      0.1190       0.000999          True
    img_densenet121    0.2801     0.0973      0.1224       0.000999          True
        probe_image    0.2488     0.1004      0.1204       0.000999          True
       img_resnet50    0.2188     0.1112      0.1317       0.000999          True
           zeros

## 5. Ablation study

Each row removes one component. Anything that does not change the result
within its confidence interval is not a contribution — delete it from the
method section rather than defending it.

In [9]:
ABLATIONS = [
    ("full system", "fusion_late"),
    ("without text (image only)", "img_resnet50"),
    ("without image (text only)", "tfidf_logreg"),
    ("without multi-task (probe only)", "probe_image"),
    ("without fine-tuning (zero-shot)", "zeroshot"),
    ("without learning (prior)", "prior"),
]
rows = []
for label, key in ABLATIONS:
    if key not in MODELS:
        continue
    m, lo, hi = bootstrap_ci(metric_macro_f1, Y, PRED[key])
    rows.append({"configuration": label, "model_key": key,
                 "macro_F1": round(metric_macro_f1(Y, PRED[key]), 4),
                 "CI_low": round(lo, 3), "CI_high": round(hi, 3)})
abl = pd.DataFrame(rows)
if len(abl) > 1:
    base = abl.iloc[0]["macro_F1"]
    abl["delta_vs_full"] = (abl["macro_F1"] - base).round(4)
abl.to_csv(ART / "ablation_classification.csv", index=False)
print(abl.to_string(index=False))

                  configuration    model_key  macro_F1  CI_low  CI_high  delta_vs_full
                    full system  fusion_late    0.6043   0.526    0.660         0.0000
      without text (image only) img_resnet50    0.2188   0.180    0.253        -0.3855
      without image (text only) tfidf_logreg    0.6361   0.563    0.688         0.0318
without multi-task (probe only)  probe_image    0.2488   0.207    0.284        -0.3555
without fine-tuning (zero-shot)     zeroshot    0.1598   0.137    0.182        -0.4445
       without learning (prior)        prior    0.1110   0.103    0.119        -0.4933


In [10]:
# RAG ablations come from notebook 05's sweep files.
for f in ["rag_k_sweep.csv", "generation_metrics.csv",
          "generation_metrics_fullreport.csv", "retrieval_metrics.csv"]:
    p = find_file(f)
    if p is not None:
        print(f"\n=== {f} ===")
        print(pd.read_csv(p).to_string(index=False))


=== rag_k_sweep.csv ===
 K  CE_precision  CE_recall  CE_F1  CE_exact_set  ROUGE-L  Recall@K  Precision@K
 1        0.2964     0.2889 0.2926        0.2587   0.1877    0.3677       0.3677
 2        0.2676     0.4095 0.3237        0.1467   0.2240    0.5291       0.3498
 3        0.4176     0.3460 0.3785        0.3629   0.2000    0.6278       0.3438
 5        0.4373     0.3873 0.4108        0.3745   0.2282    0.7444       0.3256
 7        0.4203     0.3683 0.3926        0.3514   0.2045    0.7982       0.3120
10        0.4259     0.3556 0.3875        0.3784   0.1972    0.8296       0.2960

=== generation_metrics.csv ===
             method  CE_F1  CE_precision  CE_recall  CE_exact_set  BLEU-1  BLEU-4  ROUGE-L  METEOR  distinct  mean_len
 consensus_fusion@K 0.4108        0.4373     0.3873        0.3745  0.1611  0.1002   0.2282  0.1890    0.0849    4.5985
    constant_normal 0.3171        0.3514     0.2889        0.3243  0.0000  0.0000   0.1590  0.1184    0.0039    3.0000
    retrieve_copy@1

### Bootstrap CI for generation metrics

Same logic applied to the clinical-efficacy F1 of the generated reports, so
Table 3 carries intervals too.

In [11]:
gen_path = find_file("generated_reports.csv")
onto_path = find_file("ontology.json")
if gen_path is not None and onto_path is not None:
    ontology = json.loads(onto_path.read_text())
    gen = pd.read_csv(gen_path).fillna("")

    def extract_labels(text):
        t = str(text).lower()
        out = set()
        for lab, spec in ontology.items():
            if any(re.search(p, t) for p in spec.get("include", [])) and \
               not any(re.search(p, t) for p in spec.get("exclude", [])):
                out.add(lab)
        return out

    ref_sets = [extract_labels(t) for t in gen["reference"]]
    methods = [c for c in gen.columns if c not in ("study_id", "reference")]

    def ce_f1(idx, hyp_sets):
        tp = fp = fn = 0
        for i in idx:
            hs, rs = hyp_sets[i], ref_sets[i]
            tp += len(hs & rs)
            fp += len(hs - rs)
            fn += len(rs - hs)
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        return 2 * p * r / (p + r) if p + r else 0.0

    rows = []
    for meth in methods:
        hyp_sets = [extract_labels(t) for t in gen[meth]]
        obs = ce_f1(range(len(gen)), hyp_sets)
        r_ = np.random.default_rng(SEED)
        boot = [ce_f1(r_.integers(0, len(gen), len(gen)), hyp_sets) for _ in range(1000)]
        rows.append({"method": meth, "CE_F1": round(obs, 4),
                     "CI_low": round(float(np.percentile(boot, 2.5)), 3),
                     "CI_high": round(float(np.percentile(boot, 97.5)), 3)})
    gen_ci = pd.DataFrame(rows).sort_values("CE_F1", ascending=False)
    gen_ci.to_csv(ART / "generation_ci.csv", index=False)
    print(gen_ci.to_string(index=False))

    # Paired bootstrap between the two best methods.
    if len(methods) >= 2:
        a, b = gen_ci.iloc[0]["method"], gen_ci.iloc[1]["method"]
        sa = [extract_labels(t) for t in gen[a]]
        sb = [extract_labels(t) for t in gen[b]]
        r_ = np.random.default_rng(SEED)
        diffs = []
        for _ in range(2000):
            idx = r_.integers(0, len(gen), len(gen))
            diffs.append(ce_f1(idx, sa) - ce_f1(idx, sb))
        diffs = np.array(diffs)
        p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
        print(f"\nPaired bootstrap: {a} - {b} = {diffs.mean():+.4f} "
              f"[{np.percentile(diffs,2.5):+.3f}, {np.percentile(diffs,97.5):+.3f}], "
              f"p = {p_two:.4f}")
        print("  -> " + ("significant" if p_two < 0.05 else
                         "NOT significant; report both as comparable"))

             method  CE_F1  CI_low  CI_high
 consensus_fusion@K 0.4108   0.358    0.472
    constant_normal 0.3171   0.264    0.374
    retrieve_copy@1 0.2813   0.230    0.326
  multitask_decoder 0.2192   0.185    0.253
random_train_report 0.1531   0.113    0.193

Paired bootstrap: consensus_fusion@K - constant_normal = +0.0943 [+0.043, +0.149], p = 0.0010
  -> significant


## 6. Explainability I — token attribution for the text model

For the linear text model, attribution is exact: the contribution of a token
is its TF-IDF value times its coefficient. No approximation, no saliency
hand-waving — a good thing to show alongside Grad-CAM, whose faithfulness is
genuinely contested in the literature.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

texts = df["findings_proc"].fillna("").values
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xall = vec.fit_transform(texts)
vocab = np.array(vec.get_feature_names_out())

top_terms = {}
for j, lab in enumerate(TARGETS):
    if Y[:, j].sum() < 3:
        continue
    clf = LogisticRegression(C=4.0, max_iter=2000, class_weight="balanced")
    clf.fit(Xall, Y[:, j])
    coef = clf.coef_[0]
    top = np.argsort(-coef)[:8]
    top_terms[lab] = [(vocab[t], round(float(coef[t]), 3)) for t in top]

for lab, terms in top_terms.items():
    print(f"{lab:26s} " + ", ".join(t for t, _ in terms))
pd.DataFrame([{"label": l, "top_terms": "|".join(t for t, _ in v)}
              for l, v in top_terms.items()]).to_csv(
    ART / "text_attribution.csv", index=False)

Normal study               normal in, is normal, normal, normal no, size normal, normal cervix, contour, shape no
Normal pregnancy           present, days, regular, pregnancy, os, weeks, single, 05
Fatty liver                increased, visualization of, visualization, vessels, enlarged in, echogenicity with, mild, brighter
Cystitis                   thickened, irregular, filled wall, wall is, irregular outline, having thickened, thickened wall, urine having
Bulky uterus               bulky in, bulky, is bulky, enlarged in, is enlarged, 11 cm, 11, enlarged
Hepatomegaly               size 15, enlarged in, liver mildly, 15 cm, 15, mildly, liver enlarged, mildly enlarged
Early pregnancy failure    01, is absent, absent, containing, 64, 03 cm, fetal pole, like structure
Significant PVR            cc pvr, mcc, cc, pvr, cc no, seen mcc, echopattern mcc, 45 cc
Pelvic collection          sac mild, mild pelvic, mild, outline uterus, thickened mm, mm irregular, cm echogenicity, ovary is
Renal cys

**Read this table critically.** If "Fatty liver" is driven by
*increased, echogenicity, bright* the model learned the radiological
description. If it is driven by *clinical, information, obesity* it learned
the referral reason, which will not generalise to a different clinic. This
check has killed more medical NLP results than any metric.

In [13]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (lab, terms) in zip(axes.ravel(), list(top_terms.items())[:6]):
    t, w = zip(*terms[::-1])
    ax.barh(t, w, color="#55a868")
    ax.set_title(lab, fontsize=10)
    ax.tick_params(labelsize=8)
for ax in axes.ravel()[len(top_terms):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIG / "text_attribution.png")
plt.show()

## 7. Explainability II — retrieval as explanation

The most clinically legible explanation this system can give is not a
heatmap: it is *"this looks like these three previous cases, and here is what
was written for them."* Radiologists reason by comparison; a case-based
explanation matches that far better than a blurred activation blob.

This costs nothing beyond what notebook 05 already computed, and is worth a
figure of its own in the paper.

In [14]:
emb_p = find_file("study_image_embeddings.npy")
gen_p = find_file("generated_reports.csv")
if emb_p is not None:
    Z = np.load(emb_p)
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
    dup = df["dup_group"].values if "dup_group" in df.columns else np.arange(len(df))
    rows = []
    for i in range(min(6, len(df))):
        pool = np.where((folds != folds[i]) & (dup != dup[i]))[0]
        sims = Z[i] @ Z[pool].T
        top = pool[np.argsort(-sims)[:3]]
        rows.append({
            "query": study_ids[i],
            "query_impression": df["impression"].iloc[i][:70],
            "neighbours": " | ".join(
                f"{study_ids[j]} ({sims[list(pool).index(j)]:.2f}): "
                f"{df['impression'].iloc[j][:45]}" for j in top),
        })
    expl = pd.DataFrame(rows)
    expl.to_csv(ART / "retrieval_explanations.csv", index=False)
    for _, r in expl.iterrows():
        print(f"\n{r['query']}: {r['query_impression']}")
        for part in r["neighbours"].split(" | "):
            print(f"    ↳ {part}")
else:
    print("Run notebook 04 for image embeddings to enable retrieval explanations.")


R1: Suggestive of Inhomogeneous uterus.
    ↳ R7 (0.93): Normal study at ultrasound.
    ↳ R126 (0.92): Normal study at ultrasound.
    ↳ R64 (0.92): Normal USG study.

R2: Right adnexal chocolate cyst.
    ↳ R114 (0.91): 1. Cholelithiasis. 2. Significant PVR.
    ↳ R105 (0.90): Normal size uterus, endometrial thickness is 
    ↳ R35 (0.90): 1. Enlarged prostate with significant PVR. 2.

R3: Normal size uterus, endometrial thickness is 7.9mm. Multiple nabothian
    ↳ R30 (0.93): Normal size uterus, endometrial thickness is 
    ↳ R105 (0.92): Normal size uterus, endometrial thickness is 
    ↳ R27 (0.91): Suggestive of 1. Cholelithiasis. 2. Cystitis.

R4: Suggestive of early pregnancy failure (Missed abortion).
    ↳ R208 (0.92): Suggestive of early pregnancy failure(Missed 
    ↳ R218 (0.92): Suggestive of early pregnancy failure(Missed 
    ↳ R146 (0.89): Bulky uterus, endometrial thickness is 21.1mm

R5: Suggestive of Enlarged right ovary with endometrioma.
    ↳ R60 (0.88): Sugges

## 8. Explainability III — Grad-CAM and MIL attention

Two complementary views of the same prediction:

- **MIL attention** answers *which frame* in the study mattered. Notebook 03
  saved these weights.
- **Grad-CAM** answers *which region* of that frame mattered.

Show them together: a Grad-CAM map on a frame the model gave 3% attention to
is meaningless, and presenting one without the other is how explainability
figures end up misleading.

In [15]:
RUN_GRADCAM = True
try:
    import torch
    import torch.nn.functional as F
    import cv2
    HAS_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    HAS_TORCH = False
    print("torch unavailable — Grad-CAM skipped, attention analysis still runs below.")

frames_p = find_file("frames.csv")
frames = pd.read_csv(frames_p) if frames_p is not None else None

if frames is not None:
    # Same stale-path issue as notebook 03: frames.csv holds the paths from
    # notebook 02's session. Re-anchor before reading any image.
    _s = Path(str(frames["frame_path"].iloc[0]))
    if not _s.exists():
        _rel = Path(_s.parent.name) / _s.name
        _cands = [Path("/kaggle/working/frames")]
        for _d in ("*/frames", "*/*/frames", "*/frames/*"):
            _cands += sorted(Path("/kaggle/input").glob(_d))
        _root = next((c for c in _cands if (c / _rel).exists()), None)
        if _root is None:
            import zipfile
            _zips = sorted(Path("/kaggle/input").rglob("frames.zip"))
            if _zips:
                _dest = Path("/kaggle/working/frames")
                _dest.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(_zips[0]) as _zf:
                    _zf.extractall(_dest)
                if (_dest / _rel).exists():
                    _root = _dest
                    print(f"Extracted frames from {_zips[0]}")
        if _root is None:
            print("Frame images not found — Grad-CAM will be skipped. "
                  "Attach notebook 02's output including the frames/ directory.")
            frames = None
        else:
            frames["frame_path"] = [
                str(_root / Path(q).parent.name / Path(q).name)
                for q in frames["frame_path"]]
            print(f"Re-anchored frame paths to {_root}")

Extracted frames from /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/frames.zip
Re-anchored frame paths to /kaggle/working/frames


In [16]:
att_files = list(ART.glob("attention_*.npy")) + \
            [p for p in Path("/kaggle/input").rglob("attention_*.npy")]
if att_files and frames is not None:
    att = np.load(att_files[0], allow_pickle=True).item()
    tag = att_files[0].stem.replace("attention_", "")
    rows = []
    for sid, a in att.items():
        fr = frames[frames["study_id"] == sid]
        if len(fr) == 0:
            continue
        w = np.asarray(a).ravel()[:len(fr)]
        if w.sum() == 0:
            continue
        w = w / w.sum()
        k = int(np.argmax(w))
        rows.append({"study_id": sid, "n_frames": len(fr),
                     "top_frame": fr.iloc[k]["frame_id"],
                     "top_weight": float(w[k]),
                     "entropy": float(-(w * np.log(w + 1e-9)).sum()),
                     "uniform_entropy": float(np.log(len(fr)))})
    adf = pd.DataFrame(rows)
    adf["concentration"] = (1 - adf["entropy"] / adf["uniform_entropy"].clip(lower=1e-6)).round(3)
    adf.to_csv(ART / "mil_attention_analysis.csv", index=False)
    print(f"MIL attention ({tag}): mean top-frame weight "
          f"{adf['top_weight'].mean():.3f}, mean concentration "
          f"{adf['concentration'].mean():.3f}")
    print("Concentration near 0 means the model weights every frame equally —\n"
          "the attention is decorative and should not be presented as evidence.")
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].hist(adf["top_weight"], bins=20, color="#4c72b0")
    ax[0].set_title("Attention mass on the top frame")
    ax[1].hist(adf["concentration"], bins=20, color="#c44e52")
    ax[1].set_title("Attention concentration (0 = uniform)")
    plt.tight_layout()
    plt.savefig(FIG / "mil_attention.png")
    plt.show()
else:
    print("No saved attention weights — run notebook 03's image models first.")

MIL attention (img_effnetv2s): mean top-frame weight 0.921, mean concentration 0.855
Concentration near 0 means the model weights every frame equally —
the attention is decorative and should not be presented as evidence.


In [17]:
if RUN_GRADCAM and HAS_TORCH and frames is not None:
    try:
        import timm

        class GradCAM:
            """Grad-CAM and Grad-CAM++ for a CNN feature extractor."""

            def __init__(self, model, layer):
                self.model = model.eval()
                self.acts = self.grads = None
                layer.register_forward_hook(
                    lambda m, i, o: setattr(self, "acts", o))
                layer.register_full_backward_hook(
                    lambda m, gi, go: setattr(self, "grads", go[0]))

            def __call__(self, x, class_idx, plus_plus=False):
                out = self.model(x)
                score = out[0, class_idx]
                self.model.zero_grad()
                score.backward(retain_graph=True)
                A, G = self.acts[0], self.grads[0]
                if plus_plus:
                    g2, g3 = G ** 2, G ** 3
                    denom = 2 * g2 + (A * g3).sum((1, 2), keepdim=True)
                    alpha = g2 / denom.clamp(min=1e-8)
                    w = (alpha * F.relu(G)).sum((1, 2))
                else:
                    w = G.mean((1, 2))
                cam = F.relu((w[:, None, None] * A).sum(0))
                cam = cam - cam.min()
                return (cam / cam.max().clamp(min=1e-8)).detach().cpu().numpy()

        ckpts = sorted(ART.glob("img_*_fold0.pt")) + \
                [p for p in Path("/kaggle/input").rglob("img_*_fold0.pt")]
        if not ckpts:
            print("No image-model checkpoint found — skipping Grad-CAM.")
        else:
            ck = ckpts[0]
            backbone = {"img_resnet50": "resnet50",
                        "img_densenet121": "densenet121",
                        "img_effnetv2s": "tf_efficientnetv2_s"}.get(
                            ck.stem.rsplit("_fold", 1)[0], "resnet50")
            enc = timm.create_model(backbone, pretrained=False, num_classes=0,
                                    global_pool="avg")
            sd = torch.load(ck, map_location="cpu")
            enc.load_state_dict({k[4:]: v for k, v in sd.items()
                                 if k.startswith("enc.")}, strict=False)
            head_w = sd.get("head.weight")
            head_b = sd.get("head.bias")

            class FrameClassifier(torch.nn.Module):
                def __init__(self, enc, w, b):
                    super().__init__()
                    self.enc = enc
                    self.head = torch.nn.Linear(w.shape[1], w.shape[0])
                    self.head.weight.data, self.head.bias.data = w, b

                def forward(self, x):
                    return self.head(self.enc(x))

            model = FrameClassifier(enc, head_w, head_b).to(DEVICE)
            target_layer = [m for m in model.enc.modules()
                            if isinstance(m, torch.nn.Conv2d)][-1]
            cam = GradCAM(model, target_layer)

            MEAN = np.array([0.485, 0.456, 0.406], np.float32)
            STD = np.array([0.229, 0.224, 0.225], np.float32)

            interesting = df[df["primary_label"].isin(
                ["Fatty liver", "Cholelithiasis", "Normal study"])].head(3)
            fig, axes = plt.subplots(len(interesting), 3, figsize=(10, 3.2 * len(interesting)))
            axes = np.atleast_2d(axes)
            for r, (_, row) in enumerate(interesting.iterrows()):
                fr = frames[frames["study_id"] == row["study_id"]]
                if len(fr) == 0:
                    continue
                path = fr.iloc[0]["frame_path"]
                img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))
                x = torch.tensor(((img / 255.0 - MEAN) / STD).transpose(2, 0, 1)
                                 )[None].float().to(DEVICE)
                j = TARGETS.index(row["primary_label"]) if row["primary_label"] in TARGETS else 0
                m1 = cv2.resize(cam(x, j, False), (224, 224))
                m2 = cv2.resize(cam(x, j, True), (224, 224))
                axes[r, 0].imshow(img)
                axes[r, 0].set_title(f"{row['study_id']} — {row['primary_label']}", fontsize=9)
                axes[r, 1].imshow(img)
                axes[r, 1].imshow(m1, cmap="jet", alpha=0.45)
                axes[r, 1].set_title("Grad-CAM", fontsize=9)
                axes[r, 2].imshow(img)
                axes[r, 2].imshow(m2, cmap="jet", alpha=0.45)
                axes[r, 2].set_title("Grad-CAM++", fontsize=9)
                for c in range(3):
                    axes[r, c].axis("off")
            plt.tight_layout()
            plt.savefig(FIG / "gradcam.png")
            plt.show()
    except Exception as e:
        print("Grad-CAM skipped:", type(e).__name__, e)

### A sanity check you should run before believing any heatmap

Re-run Grad-CAM with **randomly initialised** weights. If the maps look
similar to the trained ones, they are reflecting image edges and the sector
boundary, not learned pathology (Adebayo et al., 2018 — the "sanity checks
for saliency maps" result). This is a two-minute experiment that pre-empts a
reviewer objection, and a negative result here is worth reporting.

## 9. Paper-ready outputs

Everything a results section needs, in one place.

In [18]:
tables = {
    "Table 2 — classification with CI": "results_with_ci.csv",
    "Table 3 — report generation": "generation_metrics.csv",
    "Table 3b — full report track": "generation_metrics_fullreport.csv",
    "Table 4 — retrieval": "retrieval_metrics.csv",
    "Table 5 — ablation (classification)": "ablation_classification.csv",
    "Table 6 — ablation (K in RAG)": "rag_k_sweep.csv",
    "Table 7 — McNemar": "mcnemar_tests.csv",
    "Table 8 — permutation tests": "permutation_tests.csv",
    "Table 9 — generation CI": "generation_ci.csv",
    "Supp — per-class F1": "baseline_per_class.csv",
    "Supp — zero-shot AUC": "zeroshot_auc.csv",
    "Supp — embedding separability": "embedding_separability.csv",
    "Supp — text attribution": "text_attribution.csv",
    "Supp — case studies": "rag_case_studies.csv",
}
manifest = []
for caption, fname in tables.items():
    p = find_file(fname)
    manifest.append({"caption": caption, "file": fname,
                     "status": "ready" if p else "missing — run the earlier notebook"})
man = pd.DataFrame(manifest)
man.to_csv(ART / "paper_tables_manifest.csv", index=False)
print(man.to_string(index=False))

                            caption                              file status
   Table 2 — classification with CI               results_with_ci.csv  ready
        Table 3 — report generation            generation_metrics.csv  ready
       Table 3b — full report track generation_metrics_fullreport.csv  ready
                Table 4 — retrieval             retrieval_metrics.csv  ready
Table 5 — ablation (classification)       ablation_classification.csv  ready
      Table 6 — ablation (K in RAG)                   rag_k_sweep.csv  ready
                  Table 7 — McNemar                 mcnemar_tests.csv  ready
        Table 8 — permutation tests             permutation_tests.csv  ready
            Table 9 — generation CI                 generation_ci.csv  ready
                Supp — per-class F1            baseline_per_class.csv  ready
               Supp — zero-shot AUC                  zeroshot_auc.csv  ready
      Supp — embedding separability        embedding_separability.csv  ready

In [19]:
# LaTeX for the main results table.
if (ART / "results_with_ci.csv").exists():
    t = pd.read_csv(ART / "results_with_ci.csv")
    latex = t.to_latex(index=False, escape=True, float_format="%.3f",
                       column_format="l" + "r" * (t.shape[1] - 1),
                       caption="Multi-label classification performance with 95\\% "
                               "bootstrap confidence intervals (5-fold cross-validation, "
                               "n=259 studies).",
                       label="tab:classification")
    (ART / "table_classification.tex").write_text(latex)
    print(latex[:1200])

\begin{table}
\caption{Multi-label classification performance with 95\% bootstrap confidence intervals (5-fold cross-validation, n=259 studies).}
\label{tab:classification}
\begin{tabular}{lrrrrrr}
\toprule
model & micro\_F1 & micro\_F1\_95CI & macro\_F1 & macro\_F1\_95CI & macro\_AUC & macro\_AUC\_95CI \\
\midrule
multitask & 0.835 & [0.788, 0.878] & 0.797 & [0.703, 0.853] & 0.945 & [0.909, 0.975] \\
tfidf\_logreg & 0.671 & [0.626, 0.718] & 0.636 & [0.563, 0.688] & 0.923 & [0.898, 0.947] \\
fusion\_late & 0.633 & [0.588, 0.677] & 0.604 & [0.526, 0.660] & 0.906 & [0.875, 0.934] \\
tfidf\_svm & 0.711 & [0.663, 0.757] & 0.584 & [0.522, 0.633] & 0.890 & [0.856, 0.925] \\
clinicalbert & 0.474 & [0.432, 0.514] & 0.450 & [0.394, 0.491] & 0.855 & [0.816, 0.892] \\
probe\_text\_findings & 0.340 & [0.310, 0.371] & 0.335 & [0.288, 0.372] & 0.798 & [0.761, 0.830] \\
img\_effnetv2s & 0.391 & [0.347, 0.434] & 0.293 & [0.240, 0.335] & 0.700 & [0.653, 0.747] \\
probe\_concat & 0.323 & [0.289, 0.359] 

In [20]:
figs = sorted(FIG.glob("*.png"))
print(f"{len(figs)} figures in {FIG}:")
for f in figs:
    print("  ", f.name)

final = {
    "n_studies": int(len(df)),
    "n_targets": len(TARGETS),
    "models_compared": list(MODELS),
    "best_model": ci_df.iloc[0]["model"],
    "best_macro_F1": float(ci_df.iloc[0]["macro_F1"]),
    "best_macro_F1_CI": ci_df.iloc[0]["macro_F1_95CI"],
    "models_above_chance": perm_df[perm_df["above_chance"]]["model"].tolist(),
    "n_figures": len(figs),
}
(ART / "final_summary.json").write_text(json.dumps(final, indent=2))
print(json.dumps(final, indent=2))

4 figures in /kaggle/working/artifacts/figures:
   gradcam.png
   mil_attention.png
   results_with_ci.png
   text_attribution.png
{
  "n_studies": 259,
  "n_targets": 13,
  "models_compared": [
    "clinicalbert",
    "fusion_late",
    "img_densenet121",
    "img_effnetv2s",
    "img_resnet50",
    "majority",
    "prior",
    "tfidf_logreg",
    "tfidf_svm",
    "multitask",
    "probe_concat",
    "probe_image",
    "probe_text_findings",
    "zeroshot"
  ],
  "best_model": "multitask",
  "best_macro_F1": 0.797,
  "best_macro_F1_CI": "[0.703, 0.853]",
  "models_above_chance": [
    "multitask",
    "tfidf_logreg",
    "fusion_late",
    "tfidf_svm",
    "clinicalbert",
    "probe_text_findings",
    "img_effnetv2s",
    "probe_concat",
    "img_densenet121",
    "probe_image",
    "img_resnet50",
    "zeroshot"
  ],
  "n_figures": 4
}


## Writing the paper

**The sentence to build the discussion around:** state the effect, the
interval, and the limitation in one breath. For example — *"Retrieval-
augmented fusion improved clinical-efficacy F1 from 0.51 (copy@1) to 0.54
(95% CI ...), though the paired bootstrap does not establish significance at
n=259; the ranking was consistent across all five folds."*

**Limitations to state yourself, before a reviewer does:**

1. 259 studies from a single centre, one reporting style, one scanner class.
2. Labels are rule-extracted from impressions, not adjudicated by a
   radiologist. Report your `label_audit.csv` correction rate.
3. Images are photographs of console output, not DICOM — with burned-in
   annotations and no pixel spacing.
4. Findings are highly imbalanced; several classes have fewer than 10 cases
   and were excluded from the classification target.
5. No external validation set.

**Realistic venues.** With 259 studies, a Q1 journal is a stretch on novelty
alone — the honest path is either to grow the dataset substantially, or to
frame the contribution as a *method and evaluation protocol* for small-data
ultrasound report generation and target a strong applied venue
(IEEE JBHI, Computers in Biology and Medicine, Artificial Intelligence in
Medicine) or a MICCAI workshop. Reviewers there will engage with careful
small-data methodology; they will not accept an over-claimed clinical result.